In [1]:
import os
import sys
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

import importlib.util

PARQUET_DIR = "/Users/yuliia/Documents/Fraud-Detection/parquet"
fraud_ids_path = os.path.join(PARQUET_DIR, "fraud_ids.py")
spec = importlib.util.spec_from_file_location("fraud_ids", fraud_ids_path)
fraud_ids_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fraud_ids_mod)
FRAUD_WAITER_WEEK_IDS = fraud_ids_mod.FRAUD_WAITER_WEEK_IDS

PROCESSED_PATH = os.path.join(PARQUET_DIR, "processed_transactions.parquet")
WAITER_MONTH_FEATURES_PATH = os.path.join(PARQUET_DIR, "waiter_month_features.parquet")

ROOT = os.path.dirname(PARQUET_DIR)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


def build_waiter_month_features(df: pd.DataFrame) -> pd.DataFrame:
    """Per waiter_month aggregates (same logic as waiter_week_level_features, monthly grain)."""
    from config import FRAUD_IDS

    df = df.copy()
    df["fraud_person_id"] = df["person_id"].isin(FRAUD_IDS)
    df["fraud_waiter_week"] = df["waiter_week"].isin(FRAUD_WAITER_WEEK_IDS)

    waiter_share = df.groupby(["person_id", "waiter_id"]).size().reset_index(name="cnt")
    top_waiter_share = (
        waiter_share.sort_values(["person_id", "cnt"], ascending=[True, False])
        .groupby("person_id")
        .first()
        .reset_index()
        .rename(columns={"waiter_id": "top_waiter_id"})
    )
    df = df.merge(top_waiter_share[["person_id", "top_waiter_id"]], on="person_id", how="left")
    _match_top = df["waiter_id"].eq(df["top_waiter_id"]) & df["top_waiter_id"].notna()

    waiter_month_data = df.groupby("waiter_month").agg(
        waiter_id=("waiter_id", "first"),
        num_of_trn=("trn_id", "nunique"),
        unique_persons=("person_id", "nunique"),
        is_fraud=("fraud_waiter_week", "any"),
        working_days=("date", "nunique"),
        place_id=("place_id", "first"),
        month=("month", "first"),
        bonusses_accum=("bonusses_accum", "sum"),
        bonusses_used=("bonusses_used", "sum"),
        bonusses_trn=("bonus_used_flag", "sum"),
        mean_check=("gross_amount", "mean"),
        new_clients=("trn_count_before", lambda x: (x == 0).sum()),
        num_of_fraud_trn=("fraud_person_id", "sum"),
    ).reset_index()

    _top_waiter_month = (
        df.loc[_match_top]
        .groupby("waiter_month", as_index=False)
        .agg(
            persons_top_waiter_match=("person_id", "nunique"),
            trn_top_waiter_match=("trn_id", "nunique"),
        )
    )
    waiter_month_data = waiter_month_data.merge(_top_waiter_month, on="waiter_month", how="left")
    waiter_month_data["persons_top_waiter_match"] = waiter_month_data[
        "persons_top_waiter_match"
    ].fillna(0)
    waiter_month_data["trn_top_waiter_match"] = waiter_month_data["trn_top_waiter_match"].fillna(
        0
    )

    non_loyal_trn = pd.read_csv("/Users/yuliia/Documents/Fraud-Detection/non_loyal_customers.csv")
    non_loyal_trn["waiter_id"] = (
        non_loyal_trn["place_id"].astype("string").fillna("")
        + "_"
        + non_loyal_trn["trn_reg_user_code"].astype("string").fillna("")
    )
    non_loyal_trn["trn_date"] = pd.to_datetime(non_loyal_trn["trn_date"])
    non_loyal_trn["month"] = non_loyal_trn["trn_date"].dt.to_period("M").dt.start_time
    non_loyal_trn = non_loyal_trn.rename(columns={"transactions_count": "trn_count_nonloyal"})
    non_loyal_trn["waiter_month"] = (
        non_loyal_trn["waiter_id"] + "_" + non_loyal_trn["month"].astype(str)
    )

    non_loyal_trn = non_loyal_trn.groupby("waiter_month")["trn_count_nonloyal"].sum()

    waiter_month_data = waiter_month_data.merge(non_loyal_trn, on="waiter_month", how="left")

    waiter_month_data = waiter_month_data.dropna(subset=["trn_count_nonloyal"])

    waiter_month_data["trn_count_all"] = (
        waiter_month_data["num_of_trn"] + waiter_month_data["trn_count_nonloyal"]
    )
    waiter_month_data["share_loyal_trn"] = (
        waiter_month_data["num_of_trn"] / waiter_month_data["trn_count_all"]
    )

    waiter_month_data["share_trn_top_waiter"] = np.where(
        waiter_month_data["num_of_trn"] > 0,
        waiter_month_data["trn_top_waiter_match"] / waiter_month_data["num_of_trn"],
        np.nan,
    )
    waiter_month_data["share_clients_top_waiter"] = np.where(
        waiter_month_data["unique_persons"] > 0,
        waiter_month_data["persons_top_waiter_match"] / waiter_month_data["unique_persons"],
        np.nan,
    )

    waiter_month_data["trn_per_person"] = (
        waiter_month_data["num_of_trn"] / waiter_month_data["unique_persons"]
    )
    waiter_month_data["trn_per_person_norm"] = (
        waiter_month_data["trn_per_person"]
        / waiter_month_data.groupby(["place_id", "month"])["trn_per_person"].transform("median")
    )

    waiter_month_data["bonusses_used_norm_w"] = (
        waiter_month_data["bonusses_used"]
        / waiter_month_data.groupby(["place_id", "month"])["bonusses_used"].transform("median")
    )
    waiter_month_data["bonusses_used_norm_l"] = (
        waiter_month_data["bonusses_used"]
        / waiter_month_data.groupby(["place_id"])["bonusses_used"].transform("median")
    )

    waiter_month_data["share_bonusses_trn"] = (
        waiter_month_data["bonusses_trn"] / waiter_month_data["num_of_trn"]
    )
    waiter_month_data["share_bonusses_trn_norm"] = (
        waiter_month_data["share_bonusses_trn"]
        / waiter_month_data.groupby(["place_id", "month"])["share_bonusses_trn"].transform(
            "median"
        )
    )

    waiter_month_data["bonuses_used_to_accum"] = (
        waiter_month_data["bonusses_used"] / waiter_month_data["bonusses_accum"]
    )
    waiter_month_data["bonuses_used_to_accum_norm"] = (
        waiter_month_data["bonuses_used_to_accum"]
        / waiter_month_data.groupby(["place_id", "month"])["bonuses_used_to_accum"].transform(
            "median"
        )
    )

    waiter_month_data["mean_check_norm"] = (
        waiter_month_data["mean_check"]
        / waiter_month_data.groupby(["place_id", "month"])["mean_check"].transform("median")
    )

    waiter_month_data["share_new_clients"] = (
        waiter_month_data["new_clients"] / waiter_month_data["unique_persons"]
    )
    waiter_month_data["share_new_clients_norm"] = (
        waiter_month_data["share_new_clients"]
        / waiter_month_data.groupby(["place_id", "month"])["share_new_clients"].transform("median")
    )
    waiter_month_data["new_clients_norm"] = (
        waiter_month_data["new_clients"]
        / waiter_month_data.groupby(["place_id", "month"])["new_clients"].transform("median")
    )

    waiter_month_data["trn_per_day"] = (
        waiter_month_data["num_of_trn"] / waiter_month_data["working_days"]
    )
    waiter_month_data["trn_per_day_norm"] = (
        waiter_month_data["trn_per_day"]
        / waiter_month_data.groupby(["place_id", "month"])["trn_per_day"].transform("median")
    )

    waiter_month_data["unique_clients_per_day"] = (
        waiter_month_data["unique_persons"] / waiter_month_data["working_days"]
    )
    waiter_month_data["unique_clients_per_day_norm"] = (
        waiter_month_data["unique_clients_per_day"]
        / waiter_month_data.groupby(["place_id", "month"])["unique_clients_per_day"].transform(
            "median"
        )
    )

    client_trn = (
        df.groupby(["waiter_month", "person_id"])
        .agg(trn_per_client=("trn_id", "nunique"))
        .reset_index()
    )
    top_client = (
        client_trn.groupby("waiter_month")["trn_per_client"]
        .max()
        .reset_index()
        .rename(columns={"trn_per_client": "top1_client_trn"})
    )
    waiter_month_data = waiter_month_data.merge(top_client, on="waiter_month", how="left")

    waiter_month_data["top1_client_share"] = (
        waiter_month_data["top1_client_trn"] / waiter_month_data["num_of_trn"]
    )
    waiter_month_data["top1_client_share_norm"] = (
        waiter_month_data["top1_client_share"]
        / waiter_month_data.groupby(["place_id", "month"])["top1_client_share"].transform("median")
    )

    place_month = df.groupby(["place_id", "month"]).agg(
        place_num_of_waiters=("waiter_id", "nunique"),
        place_num_of_clients=("person_id", "nunique"),
        place_num_of_trn=("trn_id", "nunique"),
        total_working_days=("date", "count"),
    )
    waiter_month_data = waiter_month_data.merge(place_month, on=["place_id", "month"], how="left")

    waiter_month_data["share_of_clients"] = (
        waiter_month_data["unique_persons"] / waiter_month_data["place_num_of_clients"]
    )
    waiter_month_data["share_of_trn"] = (
        waiter_month_data["num_of_trn"] / waiter_month_data["place_num_of_trn"]
    )
    waiter_month_data["expected_share_of_trn"] = (
        waiter_month_data["working_days"] / waiter_month_data["total_working_days"]
    )
    waiter_month_data["diff_share_of_trn"] = (
        waiter_month_data["share_of_trn"] - waiter_month_data["expected_share_of_trn"]
    )
    waiter_month_data["share_unique_clients"] = (
        waiter_month_data["unique_clients_per_day"] / waiter_month_data["num_of_trn"]
    )

    # add perv / next month join keys (same waiter_id, shifted month)
    waiter_month_data["perv_month"] = waiter_month_data["month"] - pd.DateOffset(months=1)
    waiter_month_data["waiter_month_perv"] = (
        waiter_month_data["waiter_id"].astype(str)
        + "_"
        + waiter_month_data["perv_month"].astype(str)
    )
    waiter_month_data["next_month"] = waiter_month_data["month"] + pd.DateOffset(months=1)
    waiter_month_data["waiter_month_next"] = (
        waiter_month_data["waiter_id"].astype(str)
        + "_"
        + waiter_month_data["next_month"].astype(str)
    )

    waiter_month_data.drop(
        columns=[
            "perv_month",
            "next_month",
            "place_id",
            "month",
            "place_num_of_clients",
            "place_num_of_trn",
            "total_working_days",
            "expected_share_of_trn",
        ],
        inplace=True,
    )

    _pre_merge = waiter_month_data.copy()
    _prev = _pre_merge.drop(columns=["waiter_month_perv", "waiter_month_next"]).copy()
    _prev_feats = _prev.drop(columns=["waiter_month"]).add_suffix("_perv")
    _prev_feats.insert(0, "_waiter_month_join", _prev["waiter_month"].values)
    waiter_month_data = waiter_month_data.merge(
        _prev_feats,
        left_on="waiter_month_perv",
        right_on="_waiter_month_join",
        how="left",
    ).drop(columns=["_waiter_month_join"])

    _next_snap = _pre_merge.drop(columns=["waiter_month_perv", "waiter_month_next"]).copy()
    _next_feats = _next_snap.drop(columns=["waiter_month"]).add_suffix("_next")
    _next_feats.insert(0, "_waiter_month_join_next", _next_snap["waiter_month"].values)
    waiter_month_data = waiter_month_data.merge(
        _next_feats,
        left_on="waiter_month_next",
        right_on="_waiter_month_join_next",
        how="left",
    ).drop(columns=["_waiter_month_join_next"])

    # No prev/next row: use same value as current-month feature (baseline for diffs)
    for _c in list(waiter_month_data.columns):
        if not _c.endswith("_perv") or _c == "waiter_month_perv":
            continue
        _base = _c[: -len("_perv")]
        if _base in waiter_month_data.columns:
            waiter_month_data[_c] = waiter_month_data[_c].fillna(waiter_month_data[_base])
    for _c in list(waiter_month_data.columns):
        if not _c.endswith("_next") or _c == "waiter_month_next":
            continue
        _base = _c[: -len("_next")]
        if _base in waiter_month_data.columns:
            waiter_month_data[_c] = waiter_month_data[_c].fillna(waiter_month_data[_base])

    _perv_suffix = "_perv"
    for _c in list(waiter_month_data.columns):
        if not _c.endswith(_perv_suffix) or _c == "waiter_month_perv":
            continue
        _base = _c[: -len(_perv_suffix)]
        if _base not in waiter_month_data.columns:
            continue
        _s = waiter_month_data[_base]
        _p = waiter_month_data[_c]
        if pd.api.types.is_bool_dtype(_s) or pd.api.types.is_bool_dtype(_p):
            continue
        if not pd.api.types.is_numeric_dtype(_s):
            continue
        _s_f = _s.astype("float64")
        _p_f = _p.astype("float64")
        waiter_month_data[f"{_base}_diff_prev"] = _s_f - _p_f

        # (perv - current) / current — e.g. 30 vs 24 -> (24-30)/30 = -0.2
        _denom = _s_f.replace(0, np.nan)
        waiter_month_data[f"{_base}_perc_diff_prev"] = (_p_f - _s_f) / _denom

    _next_suffix = "_next"
    for _c in list(waiter_month_data.columns):
        if not _c.endswith(_next_suffix) or _c == "waiter_month_next":
            continue
        _base = _c[: -len(_next_suffix)]
        if _base not in waiter_month_data.columns:
            continue
        _s = waiter_month_data[_base]
        _n = waiter_month_data[_c]
        if pd.api.types.is_bool_dtype(_s) or pd.api.types.is_bool_dtype(_n):
            continue
        if not pd.api.types.is_numeric_dtype(_s):
            continue
        _s_f = _s.astype("float64")
        _n_f = _n.astype("float64")
        waiter_month_data[f"{_base}_diff_next"] = _s_f - _n_f
        _denom = _s_f.replace(0, np.nan)
        waiter_month_data[f"{_base}_perc_diff_next"] = (_n_f - _s_f) / _denom

    _abs_diff_cols = [
        c
        for c in waiter_month_data.columns
        if c.endswith("_diff_prev") and not c.endswith("_perc_diff_prev")
    ]
    _perc_diff_cols = [c for c in waiter_month_data.columns if c.endswith("_perc_diff_prev")]
    _abs_diff_next_cols = [
        c
        for c in waiter_month_data.columns
        if c.endswith("_diff_next") and not c.endswith("_perc_diff_next")
    ]
    _perc_diff_next_cols = [
        c for c in waiter_month_data.columns if c.endswith("_perc_diff_next")
    ]
    if _abs_diff_cols:
        waiter_month_data[_abs_diff_cols] = waiter_month_data[_abs_diff_cols].fillna(0)
    if _perc_diff_cols:
        waiter_month_data[_perc_diff_cols] = waiter_month_data[_perc_diff_cols].fillna(0)
    if _abs_diff_next_cols:
        waiter_month_data[_abs_diff_next_cols] = waiter_month_data[_abs_diff_next_cols].fillna(0)
    if _perc_diff_next_cols:
        waiter_month_data[_perc_diff_next_cols] = waiter_month_data[_perc_diff_next_cols].fillna(0)

    _diff_perc_cols = (
        _abs_diff_cols + _perc_diff_cols + _abs_diff_next_cols + _perc_diff_next_cols
    )
    if _diff_perc_cols:
        waiter_month_data[_diff_perc_cols] = waiter_month_data[_diff_perc_cols].abs()

    _drop_perv_next = [
        c
        for c in waiter_month_data.columns
        if c.endswith("_perv")
        or (
            c.endswith("_next")
            and not c.endswith("_diff_next")
            and not c.endswith("_perc_diff_next")
        )
    ]
    if _drop_perv_next:
        waiter_month_data.drop(columns=_drop_perv_next, inplace=True)

    return waiter_month_data


df = pd.read_parquet(PROCESSED_PATH, engine="pyarrow")

if os.path.exists(WAITER_MONTH_FEATURES_PATH):
    waiter_month_data = pd.read_parquet(WAITER_MONTH_FEATURES_PATH, engine="pyarrow")
else:
    waiter_month_data = build_waiter_month_features(df)
    waiter_month_data.to_parquet(WAITER_MONTH_FEATURES_PATH, index=False, engine="pyarrow")
waiter_month_data = build_waiter_month_features(df)
waiter_month_data

,waiter_month,waiter_id,num_of_trn,unique_persons,is_fraud,working_days,bonusses_accum,bonusses_used,bonusses_trn,mean_check,...,place_num_of_waiters_diff_next,place_num_of_waiters_perc_diff_next,share_of_clients_diff_next,share_of_clients_perc_diff_next,share_of_trn_diff_next,share_of_trn_perc_diff_next,diff_share_of_trn_diff_next,diff_share_of_trn_perc_diff_next,share_unique_clients_diff_next,share_unique_clients_perc_diff_next
0,539f5da6c76b7cd7fa3d859c_0.31_2025-01-01,539f5da6c76b7cd7fa3d859c_0.31,2,2,False,2,8,265,1,176.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,539f5da6c76b7cd7fa3d859c_00131_2024-09-01,539f5da6c76b7cd7fa3d859c_00131,2,2,False,2,835,0,0,3144.600000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,539f5da6c76b7cd7fa3d859c_001_2024-01-01,539f5da6c76b7cd7fa3d859c_001,95,84,False,15,8009,2637,21,874.906000,...,0.0,0.000000,0.006643,0.118786,0.004530,0.080828,0.007227,0.153129,0.009562,0.162217
3,539f5da6c76b7cd7fa3d859c_001_2024-02-01,539f5da6c76b7cd7fa3d859c_001,128,114,False,13,11458,2314,17,886.523516,...,2.0,0.111111,0.041589,0.664702,0.036725,0.606257,0.036732,0.674918,0.019192,0.280134
4,539f5da6c76b7cd7fa3d859c_001_2024-03-01,539f5da6c76b7cd7fa3d859c_001,285,253,False,18,26039,6996,34,907.156772,...,0.0,0.000000,0.005857,0.056229,0.008263,0.084916,0.005132,0.056302,0.004887,0.099084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23076,675ac8c769a6f18612504814_15543_2025-12-01,675ac8c769a6f18612504814_15543,34,29,False,3,779,414,10,228.017647,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
23077,675ac8c769a6f18612504814_16677_2025-12-01,675ac8c769a6f18612504814_16677,1,1,False,1,9,0,0,99.900000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
23078,675ac8c769a6f18612504814_17494_2025-12-01,675ac8c769a6f18612504814_17494,107,86,False,7,2644,2337,41,252.641121,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
23079,675ac8c769a6f18612504814_18352_2025-12-01,675ac8c769a6f18612504814_18352,89,83,False,5,1993,1704,27,248.056180,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [2]:
waiter_month_data[
    [
        "persons_top_waiter_match",
        "trn_top_waiter_match",
        "share_trn_top_waiter",
        "share_clients_top_waiter",
    ]
]

,persons_top_waiter_match,trn_top_waiter_match,share_trn_top_waiter,share_clients_top_waiter
0,0.0,0.0,0.000000,0.000000
1,1.0,1.0,0.500000,0.500000
2,37.0,47.0,0.494737,0.440476
3,59.0,71.0,0.554688,0.517544
4,138.0,164.0,0.575439,0.545455
...,...,...,...,...
23076,4.0,6.0,0.176471,0.137931
23077,0.0,0.0,0.000000,0.000000
23078,8.0,14.0,0.130841,0.093023
23079,6.0,7.0,0.078652,0.072289
